# ML2025 Homework 1 - Retrieval Augmented Generation with Agents

## Environment Setup

First, we will mount your own Google Drive and change the working directory.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Change the working directory to somewhere in your Google Drive.
# You could check the path by right clicking on the folder.
%cd /content/drive/MyDrive/Colab-Notebooks/GenAI_ML_NTU_Tutorial

/content/drive/MyDrive/Colab-Notebooks/GenAI_ML_NTU_Tutorial


In this section, we install the necessary python packages and download model weights of the quantized version of LLaMA 3.1 8B. Also, download the dataset. Note that the model weight is around 8GB. If you are using your Google Drive as the working directory, make sure you have enough space for the model.

In [3]:
!python3 -m pip install --no-cache-dir llama-cpp-python==0.3.4 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!python3 -m pip install googlesearch-python bs4 charset-normalizer requests-html lxml_html_clean

from pathlib import Path
if not Path('./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf').exists():
    !wget https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
if not Path('./public.txt').exists():
    !wget https://www.csie.ntu.edu.tw/~ulin/public.txt
if not Path('./private.txt').exists():
    !wget https://www.csie.ntu.edu.tw/~ulin/private.txt

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.2/445.2 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 14.6 MB/s eta 0:00:00
  Created wheel for websockets: filename=websockets-10.4-cp312-cp312-linux_x86_64.whl size=107329 sha256=7d9582464d717a05c32e88e05cf0e50a9390f9a59b0747f2eab41a3e883a990e
  Stored in directory: /root/.cache/pip/wheels/80/cf/6d/5d7e4c920cb41925a178b2d2621889c520d648bab487b1d7fd
Successfully built webso

In [4]:
import torch
if not torch.cuda.is_available():
    raise Exception('You are not using the GPU runtime. Change it first or you will suffer from the super slow inference speed!')
else:
    print('You are good to go!')

You are good to go!


## Prepare the LLM and LLM utility function

By default, we will use the quantized version of LLaMA 3.1 8B. you can get full marks on this homework by using the provided LLM and LLM utility function. You can also try out different LLM models.

In the following code block, we will load the downloaded LLM model weights onto the GPU first.
Then, we implemented the generate_response() function so that you can get the generated response from the LLM model more easily.

You can ignore "llama_new_context_with_model: n_ctx_per_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized" warning.

In [5]:
from llama_cpp import Llama

# Load the model onto GPU
llama3 = Llama(
    "./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
    verbose=False,
    n_gpu_layers=-1,
    n_ctx=16384,    # This argument is how many tokens the model can take. The longer the better, but it will consume more memory. 16384 is a proper value for a GPU with 16GB VRAM.
)

def generate_response(_model: Llama, _messages: str) -> str:
    '''
    This function will inference the model with given messages.
    '''
    _output = _model.create_chat_completion(
        _messages,
        stop=["<|eot_id|>", "<|end_of_text|>"],
        max_tokens=512,    # This argument is how many tokens the model can generate, you can change it and observe the differences.
        temperature=0,      # This argument is the randomness of the model. 0 means no randomness. You will get the same result with the same input every time. You can try to set it to different values.
        repeat_penalty=2.0,
    )["choices"][0]["message"]["content"]
    return _output

llama_new_context_with_model: n_ctx_per_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


## Llama 模型参数配置总结

### `Llama` 模型初始化参数：

1.  **模型路径 (`'./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf'`)**
    *   指定要加载的预训练模型文件路径，通常是 `gguf` 格式的量化模型。

2.  **`verbose=False`**
    *   控制是否打印详细信息。`False` 表示抑制输出，保持终端简洁。

3.  **`n_gpu_layers=-1`**
    *   指定卸载到 GPU 的模型层数。`-1` 意味着尽可能将所有可用层加载到 GPU 以最大化性能。

4.  **`n_ctx=16384`**
    *   **上下文窗口大小**。模型可以处理的最大 token 数量（输入+输出）。值越大，模型能处理的文本越长，但会消耗更多 GPU 内存。16384 对于 16GB VRAM 的 GPU 是一个合适的数值。

### `generate_response` 函数中的 `_model.create_chat_completion` 参数：

1.  **`_messages`**
    *   对话消息历史，通常是一个字典列表，包含 `role` (如 `system`, `user`) 和 `content` (消息文本)。

2.  **`stop=["<|eot_id|>", "<|end_of_text|>"]`**
    *   停止 token 列表。当模型生成这些 token 中的任何一个时，会停止生成文本，通常用于结束对话或轮次。

3.  **`max_tokens=512`**
    *   模型在一次生成中可以产生的**最大 token 数量**。用于控制回复的长度。

4.  **`temperature=0`**
    *   控制模型生成文本的**随机性**。`0` 表示完全没有随机性，相同输入会产生相同输出。

5.  **`repeat_penalty=2.0`**
    *   惩罚机制，用于**减少模型重复生成相同词语或短语**。较高的值会更强烈地阻止重复，鼓励多样化输出。

###

## Search Tool

The TA has implemented a search tool for you to search certain keywords using Google Search. You can use this tool to search for the relevant **web pages** for the given question. The search tool can be integrated in the following sections.

In [14]:
from typing import List
from googlesearch import search as _search
from bs4 import BeautifulSoup
from charset_normalizer import detect
import asyncio
from requests_html import AsyncHTMLSession
import urllib3
urllib3.disable_warnings()

async def worker(s:AsyncHTMLSession, url:str):
    try:
        header_response = await asyncio.wait_for(s.head(url, verify=False), timeout=10)
        if 'text/html' not in header_response.headers.get('Content-Type', ''):
            return None
        r = await asyncio.wait_for(s.get(url, verify=False), timeout=10)
        return r.text
    except:
        return None

async def get_htmls(urls):
    session = AsyncHTMLSession()
    tasks = (worker(session, url) for url in urls)
    return await asyncio.gather(*tasks)

async def search(keyword: str, n_results: int=3) -> List[str]:
    '''
    This function will search the keyword and return the text content in the first n_results web pages.

    Warning: You may suffer from HTTP 429 errors if you search too many times in a period of time. This is unavoidable and you should take your own risk if you want to try search more results at once.
    The rate limit is not explicitly announced by Google, hence there's not much we can do except for changing the IP or wait until Google unban you (we don't know how long the penalty will last either).
    '''
    keyword = keyword[:100]
    # First, search the keyword and get the results. Also, get 2 times more results in case some of them are invalid.
    results = list(_search(keyword, n_results * 2, lang="zh", unique=True))
    # Then, get the HTML from the results. Also, the helper function will filter out the non-HTML urls.
    results = await get_htmls(results)
    # Filter out the None values.
    results = [x for x in results if x is not None]
    # Parse the HTML.
    results = [BeautifulSoup(x, 'html.parser') for x in results]
    # Get the text from the HTML and remove the spaces. Also, filter out the non-utf-8 encoding.
    results = [''.join(x.get_text().split()) for x in results if detect(x.encode()).get('encoding') == 'utf-8']
    # Return the first n results.
    return results[:n_results]

In [15]:
from googlesearch import search

query = "企鹅宝宝 名字"
results = list(search(query, 5))
print(results)

[]


## Test the LLM inference pipeline

In [22]:
# You can try out different questions here.
test_question='请问谁是 Taylor Swift？'

messages = [
    {"role": "system", "content": "你是 LLaMA-3.1-8B，是用来回答问题的 AI。使用中文时只会使用简体中文来回答问题。"},    # System prompt
    {"role": "user", "content": test_question}, # User prompt
]

print(generate_response(llama3, messages))

泰勒·斯威夫特（Taylor Swift）是美国的一位知名流行音乐歌手、词曲作家和演员。她出生于1989年12月13日，来自田纳西州。她的唱片风格从乡村乐逐渐转变为多样化的 Поп/摇滚。

泰勒·斯威夫特早期以其独具个性的歌词、清澈的声音和情感丰富的情绪表达而闻名。她在2006年发行了首张专辑《Taylor Swift》，并迅速获得成功。随后，她推出了多部畅销的唱片，如 《Fearless》（2010）、_1989》( 20)15)、 _reputation_(2）017)，以及最新作品如 <a href="https://zh.wikipedia.org/wiki/%E6%B3%A1%E5%8F%D4">Lover</ a>(19))和<em>Midnights </ em >（202)(22）。

泰勒·斯威夫特的音乐经常探讨爱情、友谊以及个人成长等主题。她还因其强烈的声音支持女性权利运动而受到广泛赞誉。


## Agents

The TA has implemented the Agent class for you. You can use this class to create agents that can interact with the LLM model. The Agent class has the following attributes and methods:
- Attributes:
    - role_description: The role of the agent. For example, if you want this agent to be a history expert, you can set the role_description to "You are a history expert. You will only answer questions based on what really happened in the past. Do not generate any answer if you don't have reliable sources.".
    - task_description: The task of the agent. For example, if you want this agent to answer questions only in yes/no, you can set the task_description to "Please answer the following question in yes/no. Explanations are not needed."
    - llm: Just an indicator of the LLM model used by the agent.
- Method:
    - inference: This method takes a message as input and returns the generated response from the LLM model. The message will first be formatted into proper input for the LLM model. (This is where you can set some global instructions like "Please speak in a polite manner" or "Please provide a detailed explanation".) The generated response will be returned as the output.

In [7]:
class LLMAgent():
    def __init__(self, role_description: str, task_description: str, llm:str="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"):
        self.role_description = role_description   # Role means who this agent should act like. e.g. the history expert, the manager......
        self.task_description = task_description    # Task description instructs what task should this agent solve.
        self.llm = llm  # LLM indicates which LLM backend this agent is using.
    def inference(self, message:str) -> str:
        if self.llm == 'bartowski/Meta-Llama-3.1-8B-Instruct-GGUF': # If using the default one.
            # TODO: Design the system prompt and user prompt here.
            # Format the messsages first.
            messages = [
                {"role": "system", "content": f"{self.role_description}"},  # Hint: you may want the agents to speak Traditional Chinese only.
                {"role": "user", "content": f"{self.task_description}\n{message}"}, # Hint: you may want the agents to clearly distinguish the task descriptions and the user messages. A proper seperation text rather than a simple line break is recommended.
            ]
            return generate_response(llama3, messages)
        else:
            # TODO: If you want to use LLMs other than the given one, please implement the inference part on your own.
            return ""

TODO: Design the role description and task description for each agent.

In [8]:
# TODO: Design the role and task description for each agent.

# This agent may help you filter out the irrelevant parts in question descriptions.
question_extraction_agent = LLMAgent(
    role_description="你是一个擅长理解用户问题的AI助手，能够从冗长或口语化的描述中提取核心问题。",
    task_description="""请从用户输入中提取出“真正的问题。
    要求：
    1. 去除无关的背景信息、寒暄、情绪表达
    2. 保留用户真正想问的核心内容
    3. 用一句清晰、简洁的问句表达
    4. 不要添加额外解释

    只输出提取后的问题。
    """

)

# This agent may help you extract the keywords in a question so that the search tool can find more accurate results.
keyword_extraction_agent = LLMAgent(
    role_description="你是一个信息检索优化助手，擅长从问题中提取高质量关键词。",
    task_description="""请从给定问题中提取适合用于搜索的关键词。
    要求：
    1. 提取最重要的名词或术语
    2. 去除无意义的词（如“为什么”、“怎么”、“请问”等）
    3. 关键词数量控制在3-6个
    4. 使用空格分隔关键词
    5. 不要输出完整句子

    只输出关键词。
    """,
)

# This agent is the core component that answers the question.
qa_agent = LLMAgent(
    role_description="你是 LLaMA-3.1-8B，是一个用于回答问题的AI助手。使用中文时只使用简体中文",
    task_description="""
    请根据提供的信息回答问题。

    要求：
    1. 结合给定的上下文进行回答
    2. 回答要准确、清晰、有逻辑
    3. 如果信息不足，请说明“根据现有信息无法确定”
    4. 不要编造内容
    问题是:
    """,
)

## RAG pipeline

TODO: Implement the RAG pipeline.

Please refer to the homework description slides for hints.

Also, there might be more heuristics (e.g. classifying the questions based on their lengths, determining if the question need a search or not, reconfirm the answer before returning it to the user......) that are not shown in the flow charts. You can use your creativity to come up with a better solution!

- Naive approach (simple baseline)

    ![](https://www.csie.ntu.edu.tw/~ulin/naive.png)

- Naive RAG approach (medium baseline)

    ![](https://www.csie.ntu.edu.tw/~ulin/naive_rag.png)

- RAG with agents (strong baseline)

    ![](https://www.csie.ntu.edu.tw/~ulin/rag_agent.png)

In [9]:
async def pipeline(question: str) -> str:
    # TODO: Implement your pipeline.
    # Currently, it only feeds the question directly to the LLM.
    # You may want to get the final results through multiple inferences.
    # Just a quick reminder, make sure your input length is within the limit of the model context window (16384 tokens), you may want to truncate some excessive texts.

    # Step 1: Question Extraction
    extracted_question = question_extraction_agent.inference(question)
    print(f"Extracted Question: {extracted_question}")

    # Step 2: Keyword Extraction
    keywords = keyword_extraction_agent.inference(extracted_question)
    print(f"Keywords for Search: {keywords}")

    # Step 3: Information Retrieval using Search Tool
    search_results = await search(keywords)

    context = ""
    if search_results:
        # Combine search results into a single context string, potentially truncating if too long
        # For simplicity, we'll join them. You might need more sophisticated truncation logic
        # if search_results can be very large.
        context = "\n".join(search_results)
        print(f"Retrieved Context (first 200 chars): {context[:200]}...")
    else:
        print("No relevant search results found.")

    # Step 4: Prepare prompt for QA Agent with context and question
    # Ensure the combined input length is within the model context window (16384 tokens)
    # A simple concatenation for now, consider more advanced truncation or summarization
    # if context becomes too large.
    qa_prompt = f"Context: {context}\n\nQuestion: {extracted_question}"

    # Step 5: Answer Generation using QA Agent
    final_answer = qa_agent.inference(qa_prompt)
    print(f"Final Answer: {final_answer}")

    return final_answer

## Answer the questions using your pipeline!

Since Colab has usage limit, you might encounter the disconnections. The following code will save your answer for each question. If you have mounted your Google Drive as instructed, you can just rerun the whole notebook to continue your process.

In [13]:
%cd /content/drive/MyDrive/Colab-Notebooks/GenAI_ML_NTU_Tutorial
!rm ./s4899328_*.txt

/content/drive/MyDrive/Colab-Notebooks/GenAI_ML_NTU_Tutorial


In [11]:
from pathlib import Path

# Fill in your student ID first.
STUDENT_ID = "s4899328"

STUDENT_ID = STUDENT_ID.lower()
with open('./public.txt', 'r') as input_f:
    questions = input_f.readlines()
    questions = [l.strip().split(',')[0] for l in questions]
    for id, question in enumerate(questions, 1):
        if Path(f"./{STUDENT_ID}_{id}.txt").exists():
            continue
        answer = await pipeline(question)
        answer = answer.replace('\n',' ')
        print(id, answer)
        with open(f'./{STUDENT_ID}_{id}.txt', 'w') as output_f:
            print(answer, file=output_f)

with open('./private.txt', 'r') as input_f:
    questions = input_f.readlines()
    for id, question in enumerate(questions, 31):
        if Path(f"./{STUDENT_ID}_{id}.txt").exists():
            continue
        answer = await pipeline(question)
        answer = answer.replace('\n',' ')
        print(id, answer)
        with open(f'./{STUDENT_ID}_{id}.txt', 'a') as output_f:
            print(answer, file=output_f)

Extracted Question: 哪間學校的校歌是「虎山雄風飛揚」？
Keywords for Search: 虎山雄風飛揚 校歌 中學
Searching Google for keyword: '虎山雄風飛揚 校歌 中學'
Google search returned 0 URLs: []
No URLs returned from initial Google search.
No relevant search results found.
Final Answer: 根据提供的上下文信息，我无法确定哪间学校有校歌是「虎山雄風飛揚」。
1 根据提供的上下文信息，我无法确定哪间学校有校歌是「虎山雄風飛揚」。
Extracted Question: 2025年初，NCC規定境外郵購自用產品的審查費多少錢？
Keywords for Search: NCC 郵購審查費境外郿自用
Searching Google for keyword: 'NCC 郵購審查費境外郿自用'
Google search returned 0 URLs: []
No URLs returned from initial Google search.
No relevant search results found.
Final Answer: 根据现有信息无法确定。
2 根据现有信息无法确定。
Extracted Question: 什么时候发布了第一代 iPhone？
Keywords for Search: iPhone 第一代
Searching Google for keyword: 'iPhone 第一代'
Google search returned 0 URLs: []
No URLs returned from initial Google search.
No relevant search results found.
Final Answer: 根据提供的上下文信息，第一代 iPhone于2007年发布。
3 根据提供的上下文信息，第一代 iPhone于2007年发布。
Extracted Question: 台灣大學進階英文免修申請規定中，托福網路測驗 TOEFL iBT 要達到多少分才能申请？
Keywords for Search: 托福網路測驗 TOE

In [12]:
# Combine the results into one file.
with open(f'./{STUDENT_ID}.txt', 'w') as output_f:
    for id in range(1,91):
        with open(f'./{STUDENT_ID}_{id}.txt', 'r') as input_f:
            answer = input_f.readline().strip()
            print(answer, file=output_f)